# 01 - Binary NLI Fine-Tuning (Contradiction vs Not-Contradiction)

Fine-tune `cross-encoder/nli-deberta-v3-base` for **binary** contradiction detection.

The pretrained model has 3 output classes (contradiction / entailment / neutral). We replace the
classification head with a 2-class head while keeping all encoder weights — this lets the backbone
transfer its NLI representations while the head learns our domain-specific binary task.

**Input:** `../experiments/data/processed/ContraDoc/triples.jsonl`  
**Output:** `models/nli_binary/` (saved HuggingFace model + tokenizer)  
**Labels:** `0 = not_contradiction`, `1 = contradiction`

> **Note on dataset size:** Only 37 gold contradiction pairs exist in the 100-doc sample.
> This notebook is a runnable scaffold / smoke test. A real fine-tuning run needs the full
> 891-doc ContraDoc dataset and cross-document negative mining.

In [1]:
import json
import random
from pathlib import Path

import numpy as np
import torch
from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    Trainer,
    TrainingArguments,
)

TRIPLES_PATH = Path("../experiments/data/processed/ContraDoc/triples.jsonl")
MODEL_NAME = "cross-encoder/nli-deberta-v3-base"
OUTPUT_DIR = Path("models/nli_binary")
SEED = 42
MAX_LEN = 256
BATCH_SIZE = 8
EPOCHS = 3
NEG_RATIO = 3  # negatives per positive

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"device: {device}")

c:\Users\mfaha\Envs\cuda_env\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


device: cuda


## Build dataset from triples.jsonl

**Positives** — gold evidence/ref sentence pairs from YES docs (37 pairs).  
**Negatives** — sampled from two sources at `NEG_RATIO : 1`:
- *In-domain hard negatives*: sentence pairs from YES docs where neither sentence is the evidence or ref
- *Clear negatives*: random sentence pairs from NO docs

In [2]:
docs = [json.loads(l) for l in TRIPLES_PATH.open(encoding="utf-8")]
yes_docs = [d for d in docs if d["contradiction"] == "YES"]
no_docs  = [d for d in docs if d["contradiction"] == "NO"]
print(f"YES docs: {len(yes_docs)}  NO docs: {len(no_docs)}")

positives = []
for d in yes_docs:
    sents = {s["sentence_id"]: s["source_text"] for s in d["sentences"]}
    evid_id = d["gold_evidence_sentence_id"]
    ref_ids = d["gold_ref_sentence_ids"]
    evid_text = sents.get(evid_id, "").strip()
    for ref_id in ref_ids:
        ref_text = sents.get(ref_id, "").strip()
        if evid_text and ref_text:
            positives.append({
                "text_a": evid_text,
                "text_b": ref_text,
                "label": 1,
                "doc_id": d["doc_id"],
                "contra_type": d["contra_type"],
            })

print(f"Positive pairs: {len(positives)}")

# Hard negatives from YES docs (non-gold sentence pairs)
hard_negatives = []
for d in yes_docs:
    sents = d["sentences"]
    gold_ids = {d["gold_evidence_sentence_id"]} | set(d["gold_ref_sentence_ids"])
    non_gold = [s for s in sents if s["sentence_id"] not in gold_ids and s["source_text"].strip()]
    for i in range(len(non_gold)):
        for j in range(i + 1, len(non_gold)):
            hard_negatives.append({
                "text_a": non_gold[i]["source_text"].strip(),
                "text_b": non_gold[j]["source_text"].strip(),
                "label": 0,
                "doc_id": d["doc_id"],
            })

# Clear negatives from NO docs
clear_negatives = []
for d in no_docs:
    sents = [s for s in d["sentences"] if s["source_text"].strip()]
    for i in range(len(sents)):
        for j in range(i + 1, len(sents)):
            clear_negatives.append({
                "text_a": sents[i]["source_text"].strip(),
                "text_b": sents[j]["source_text"].strip(),
                "label": 0,
                "doc_id": d["doc_id"],
            })

print(f"Hard negatives pool: {len(hard_negatives)}")
print(f"Clear negatives pool: {len(clear_negatives)}")

n_neg = len(positives) * NEG_RATIO
n_hard = min(n_neg // 2, len(hard_negatives))
n_clear = min(n_neg - n_hard, len(clear_negatives))

sampled_hard  = random.sample(hard_negatives, n_hard)
sampled_clear = random.sample(clear_negatives, n_clear)
negatives = sampled_hard + sampled_clear

all_pairs = positives + negatives
random.shuffle(all_pairs)

print(f"\nDataset: {len(positives)} pos + {len(negatives)} neg = {len(all_pairs)} total")

YES docs: 50  NO docs: 50
Positive pairs: 37
Hard negatives pool: 32519
Clear negatives pool: 42242

Dataset: 37 pos + 111 neg = 148 total


## Train / Val split

In [3]:
labels = [p["label"] for p in all_pairs]
train_pairs, val_pairs = train_test_split(
    all_pairs, test_size=0.2, stratify=labels, random_state=SEED
)

print(f"Train: {len(train_pairs)}  (pos={sum(p['label'] for p in train_pairs)}, neg={sum(1-p['label'] for p in train_pairs)})")
print(f"Val:   {len(val_pairs)}  (pos={sum(p['label'] for p in val_pairs)}, neg={sum(1-p['label'] for p in val_pairs)})")

Train: 118  (pos=30, neg=88)
Val:   30  (pos=7, neg=23)


## Architecture Change: 3-class → 2-class

`cross-encoder/nli-deberta-v3-base` is a `DebertaV2ForSequenceClassification` with `num_labels=3`.
Its classifier head is a linear layer with shape `(3, hidden_dim)`.

We load it with `num_labels=2, ignore_mismatched_sizes=True`:
- **`num_labels=2`** — tells HuggingFace to build a 2-output classifier layer `(2, hidden_dim)`
- **`ignore_mismatched_sizes=True`** — suppresses the size-mismatch error that would occur because
  the checkpoint's classifier weights are `(3, hidden_dim)` while the new head expects `(2, hidden_dim)`;
  the head is randomly re-initialized, all encoder weights load normally

Result: full pretrained DeBERTa backbone + fresh binary classification head.

## Load tokenizer + binary model

In [4]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,
    ignore_mismatched_sizes=True,
)
model.config.id2label = {0: "not_contradiction", 1: "contradiction"}
model.config.label2id = {"not_contradiction": 0, "contradiction": 1}

# Verify head is 2-class
# DebertaV2ForSequenceClassification uses a plain nn.Linear as classifier
head_weight = model.classifier.weight
print(f"Classifier head shape: {head_weight.shape}  (expect [2, hidden_dim])")
assert head_weight.shape[0] == 2, "Head is not binary!"

total_params = sum(p.numel() for p in model.parameters())
print(f"Total parameters: {total_params:,}")

Some weights of DebertaV2ForSequenceClassification were not initialized from the model checkpoint at cross-encoder/nli-deberta-v3-base and are newly initialized because the shapes did not match:
- classifier.bias: found shape torch.Size([3]) in the checkpoint and torch.Size([2]) in the model instantiated
- classifier.weight: found shape torch.Size([3, 768]) in the checkpoint and torch.Size([2, 768]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Classifier head shape: torch.Size([2, 768])  (expect [2, hidden_dim])
Total parameters: 184,423,682


## PyTorch Dataset + tokenize

In [5]:
class PairDataset(Dataset):
    def __init__(self, pairs, tokenizer, max_len):
        enc = tokenizer(
            [p["text_a"] for p in pairs],
            [p["text_b"] for p in pairs],
            truncation=True,
            padding="max_length",
            max_length=max_len,
            return_tensors="pt",
        )
        self.input_ids      = enc["input_ids"]
        self.attention_mask = enc["attention_mask"]
        self.token_type_ids = enc.get("token_type_ids")  # DeBERTa may omit this
        self.labels         = torch.tensor([p["label"] for p in pairs], dtype=torch.long)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, i):
        item = {
            "input_ids":      self.input_ids[i],
            "attention_mask": self.attention_mask[i],
            "labels":         self.labels[i],
        }
        if self.token_type_ids is not None:
            item["token_type_ids"] = self.token_type_ids[i]
        return item


train_ds = PairDataset(train_pairs, tokenizer, MAX_LEN)
val_ds   = PairDataset(val_pairs,   tokenizer, MAX_LEN)
print(f"Train dataset: {len(train_ds)} samples")
print(f"Val dataset:   {len(val_ds)} samples")

# Sanity check one sample
sample = train_ds[0]
print(f"Sample keys: {list(sample.keys())}")
print(f"input_ids shape: {sample['input_ids'].shape}")

Train dataset: 118 samples
Val dataset:   30 samples
Sample keys: ['input_ids', 'attention_mask', 'labels', 'token_type_ids']
input_ids shape: torch.Size([256])


## Train

In [6]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

training_args = TrainingArguments(
    output_dir=str(OUTPUT_DIR / "checkpoints"),
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    logging_steps=10,
    seed=SEED,
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
)

trainer.train()

 22%|██▏       | 10/45 [01:51<11:08, 19.11s/it]

{'loss': 0.515, 'grad_norm': 11.961000442504883, 'learning_rate': 3.888888888888889e-05, 'epoch': 0.67}


                                               
 33%|███▎      | 15/45 [05:36<23:53, 47.80s/it]

{'eval_loss': 0.6129023432731628, 'eval_runtime': 15.5479, 'eval_samples_per_second': 1.93, 'eval_steps_per_second': 0.257, 'epoch': 1.0}


 44%|████▍     | 20/45 [07:03<08:21, 20.08s/it]

...

                                               
100%|██████████| 45/45 [10:50<00:00,  6.89s/it]

{'eval_loss': 0.7765287756919861, 'eval_runtime': 0.8451, 'eval_samples_per_second': 35.499, 'eval_steps_per_second': 4.733, 'epoch': 3.0}


100%|██████████| 45/45 [10:55<00:00, 14.57s/it]

{'train_runtime': 655.6618, 'train_samples_per_second': 0.54, 'train_steps_per_second': 0.069, 'train_loss': 0.2606962733798557, 'epoch': 3.0}


TrainOutput(global_step=45, training_loss=0.2606962733798557, metrics={'train_runtime': 655.6618, 'train_samples_per_second': 0.54, 'train_steps_per_second': 0.069, 'total_flos': 46571491989504.0, 'train_loss': 0.2606962733798557, 'epoch': 3.0})

## Evaluate on validation set

In [7]:
preds_out = trainer.predict(val_ds)
logits = preds_out.predictions
y_pred = np.argmax(logits, axis=1)
y_true = [p["label"] for p in val_pairs]

print(classification_report(
    y_true, y_pred,
    target_names=["not_contradiction", "contradiction"],
    digits=3,
))

100%|██████████| 4/4 [00:10<00:00,  2.69s/it]


                   precision    recall  f1-score   support

not_contradiction      0.793     1.000     0.885        23
    contradiction      1.000     0.143     0.250         7

         accuracy                          0.800        30
        macro avg      0.897     0.571     0.567        30
     weighted avg      0.841     0.800     0.737        30



## Smoke test on known pairs

In [8]:
smoke_pairs = [
    # Gold contradiction from triples.jsonl
    (
        "Little Gidding is the first poem of T. S. Eliot's Four Quartets.",
        "Little Gidding is the fourth and final poem of T. S. Eliot's Four Quartets.",
    ),
    # Numeric contradiction
    (
        "Both Sébergué and Ndikert ranked first in their respective heats.",
        "Both Sébergué and Ndikert ranked seventh in their respective heats.",
    ),
    # Clearly entailing / same meaning
    (
        "The cat is sleeping on the mat.",
        "The cat is resting on the mat.",
    ),
    # Unrelated
    (
        "Paris is the capital of France.",
        "The Amazon River is the longest river in the world.",
    ),
]

model.eval()
model.to(device)

with torch.no_grad():
    enc = tokenizer(
        [p[0] for p in smoke_pairs],
        [p[1] for p in smoke_pairs],
        truncation=True,
        padding=True,
        max_length=MAX_LEN,
        return_tensors="pt",
    ).to(device)
    out = model(**enc)
    probs = torch.softmax(out.logits, dim=-1).cpu().numpy()

id2label = model.config.id2label
print(f"{'text_a[:60]':62s}  {'pred':18s}  not_contra  contra")
print("-" * 100)
for (a, b), prob in zip(smoke_pairs, probs):
    pred = id2label[int(np.argmax(prob))]
    print(f"{a[:60]:62s}  {pred:18s}  {prob[0]:.3f}       {prob[1]:.3f}")

text_a[:60]                                                     pred                not_contra  contra
----------------------------------------------------------------------------------------------------
Little Gidding is the first poem of T. S. Eliot's Four Quart    contradiction       0.403       0.597
Both Sébergué and Ndikert ranked first in their respective h    contradiction       0.360       0.640
The cat is sleeping on the mat.                                 contradiction       0.478       0.522
Paris is the capital of France.                                 not_contradiction   0.945       0.055


## Save model

In [9]:
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"Saved to {OUTPUT_DIR.resolve()}")
print("Files:", [f.name for f in OUTPUT_DIR.iterdir()])

Saved to E:\Downloads\AIT\Second Semester\Artificial Intelligence Natural Language Understanding\Project\AT82.05-Claim-Contradiction-Over-Knowledge-Graphs\fine-tuning\models\nli_binary
Files: ['checkpoints', 'config.json', 'model.safetensors', 'special_tokens_map.json', 'tokenizer.json', 'tokenizer_config.json']


## Reload sanity check

Confirm the saved model reloads correctly as a binary classifier.

In [10]:
reloaded = AutoModelForSequenceClassification.from_pretrained(OUTPUT_DIR)
print(f"Reloaded num_labels: {reloaded.config.num_labels}")
print(f"id2label: {reloaded.config.id2label}")
assert reloaded.config.num_labels == 2
print("Reload check passed.")

Reloaded num_labels: 2
id2label: {0: 'not_contradiction', 1: 'contradiction'}
Reload check passed.
